# Notebook 2 — MLP Character Model on `names.txt`

이제 `names.txt`는 그대로 두고, **dataset을 일반화**하고 **모델을 MLP로 확장**합니다.

- bigram: 현재 문자 1개만 사용
- MLP: 길이 `block_size`의 context 전체 사용
- 표현도 one-hot 대신 **embedding**을 사용

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# 인터넷에서 이름 데이터셋 다운로드
if not Path("names.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
chars = ['.'] + chars # '.'은 단어의 시작과 끝을 의미하는 특수 기호 (인덱스 0)

# 문자를 숫자로(stoi), 숫자를 문자로(itos) 바꾸는 딕셔너리 생성
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi) # 총 고유 문자 수: 27개
encoded_words = [[stoi[ch] for ch in w] for w in words]

## 1. 일반화된 context dataset

In [2]:
class NamesContextDataset(Dataset):
    def __init__(self, encoded_words, block_size):
        self.X, self.Y = [], []
        for word in encoded_words:
            # block_size가 3이므로, 처음 시작할 때 [0, 0, 0]으로 채웁니다.
            # 즉, "마침표 3개를 보고 첫 글자를 맞춰라!"라는 의미가 됩니다.
            context = [0] * block_size

            for ix in word + [0]:
                self.X.append(context.copy())
                self.Y.append(ix)

                # 핵심 부분: context를 업데이트합니다.
                # 맨 앞의 글자는 버리고(context[1:]), 방금 정답이었던 글자를 맨 뒤에 붙입니다([ix]).
                # 마치 '이동 평균선(Moving Average)'을 구할 때 윈도우(Window)를 한 칸씩 옆으로 미는 것과 같습니다.
                context = context[1:] + [ix]

        self.X = torch.tensor(self.X, dtype=torch.long)
        self.Y = torch.tensor(self.Y, dtype=torch.long)

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# 기억력을 3글자로 늘렸습니다!
block_size = 3
dataset = NamesContextDataset(encoded_words, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

xb, yb = next(iter(loader))
print("xb.shape:", xb.shape) # 결과: (64, 3) -> 64개의 데이터가 각각 3개의 글자(과거 3글자)를 가지고 있습니다.
print("yb.shape:", yb.shape) # 결과: (64,)   -> 64개의 정답(다음 글자 1개)입니다.

xb.shape: torch.Size([64, 3])
yb.shape: torch.Size([64])


## 2. MLP model

In [3]:
class MLPCharacterModel(nn.Module):
    # emb_dim: 각 알파벳을 몇 개의 숫자로 표현할 것인가? (여기선 10차원)
    # hidden_dim: 은닉층(Hidden Layer)의 크기 (여기선 200개의 노드)
    def __init__(self, vocab_size, block_size, emb_dim=10, hidden_dim=200):
        super().__init__()

        # nn.Sequential은 여러 가지 연산(Layer)을 순서대로 하나로 묶어주는 편리한 컨테이너입니다.
        # Scikit-learn의 Pipeline(파이프라인)과 거의 동일한 역할입니다.
        self.net = nn.Sequential(
            # 1. Embedding Layer: Pandas의 pd.get_dummies() 같은 원핫인코딩 대신,
            # 단어(글자)를 연속적인 실수 백터로 압축합니다. (예: 'A' -> [0.1, -0.3, 0.5, ... 총 10개])
            # 입력 데이터가 (64, 3)으로 들어오면 -> (64, 3, 10)으로 변환됩니다.
            nn.Embedding(vocab_size, emb_dim),

            # 2. Flatten: 3차원 데이터 (64, 3, 10)을 2차원 (64, 30)으로 평평하게 폅니다.
            # 글자 3개가 각각 10차원이니, 이를 한 줄로 쫙 이어 붙여서 총 30차원의 입력 변수(Feature)로 만드는 것입니다.
            nn.Flatten(),

            # 3. Linear Layer (은닉층): 선형 회귀(Y = WX + b) 연산을 수행합니다.
            # 30개의 입력 변수를 받아서, 200개의 새로운 특징(hidden_dim)으로 확장하여 뽑아냅니다.
            nn.Linear(block_size * emb_dim, hidden_dim),

            # 4. Tanh (활성화 함수): 비선형성을 부여합니다.
            # 만약 이것이 없다면 선형 계층을 아무리 깊게 쌓아도 결국 하나의 거대한 선형 회귀와 다를 바가 없어집니다.
            # 데이터의 복잡한 곡선 패턴을 잡아내기 위해 -1 ~ 1 사이로 값을 찌그러뜨리는 수학 함수입니다.
            nn.Tanh(),

            # 5. Linear Layer (출력층): 200개의 은닉 특징을 받아서,
            # 우리가 최종적으로 예측해야 하는 27개 알파벳 각각의 '점수(logits)'로 변환합니다.
            nn.Linear(hidden_dim, vocab_size),
        )

    def forward(self, x):
        # 데이터가 입력되면 위에 정의한 순서(net)대로 쭈욱 통과시켜 결과를 반환합니다.
        return self.net(x)

model = MLPCharacterModel(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape) # 결과: (64, 27) -> 64개의 문제에 대해 27개 알파벳 확률 점수 출력
print("initial loss:", F.cross_entropy(logits, yb).item())

logits.shape: torch.Size([64, 27])
initial loss: 3.3033363819122314


## 3. Train / Eval

In [4]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

# GPU가 있다면 GPU를, 없다면 CPU를 사용합니다.
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPCharacterModel(vocab_size, block_size).to(device)

# 이번에는 학습률(Learning Rate)을 0.01 (1e-2)로 설정했습니다.
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

# 데이터를 총 30번 반복해서 학습합니다.
# 이전 모델(Bigram)보다 loss 값이 훨씬 낮아지는 것(약 2.26)을 볼 수 있습니다.
# 즉, 3글자를 보니 다음 글자를 맞출 확률이 훨씬 높아졌다는 뜻입니다!
for epoch in range(30):
    train_loss = train_one_epoch(model, loader, optimizer, device)
    if epoch % 5 == 0 or epoch == 29:
        print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.3618
epoch  5 | train loss 2.2781
epoch 10 | train loss 2.2739
epoch 15 | train loss 2.2709
epoch 20 | train loss 2.2699
epoch 25 | train loss 2.2690
epoch 29 | train loss 2.2682


## 4. 학습

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPCharacterModel(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

for epoch in range(30):
    train_loss = train_one_epoch(model, loader, optimizer, device)
    if epoch % 5 == 0 or epoch == 29:
        print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.3551
epoch  5 | train loss 2.2754
epoch 10 | train loss 2.2696
epoch 15 | train loss 2.2673
epoch 20 | train loss 2.2665
epoch 25 | train loss 2.2636
epoch 29 | train loss 2.2650


## 5. Sampling

In [6]:
@torch.no_grad()
def sample(model, block_size, itos, device, num_samples=10, max_len=20):
    model.eval()
    results = []

    for _ in range(num_samples):
        # 시작할 때는 항상 마침표 3개([0, 0, 0])로 시작합니다.
        context = torch.zeros((1, block_size), dtype=torch.long, device=device)
        out = []

        for _ in range(max_len):
            logits = model(context)
            # 점수(logits)를 확률(0~1)로 바꿉니다.
            probs = F.softmax(logits, dim=-1)
            # 그 확률을 바탕으로 알파벳 하나를 무작위로 뽑습니다.
            ix = torch.multinomial(probs, num_samples=1)
            next_token = ix.item()

            if next_token == 0: # 단어의 끝(마침표)이 나오면 멈춤
                break

            out.append(itos[next_token])

            # 다음 턴을 위해 context를 업데이트합니다! (여기서 이동평균선의 윈도우를 미는 것과 같은 원리가 쓰입니다)
            # context[:, 1:] -> 첫 번째 글자는 빼고
            # ix -> 방금 뽑은 글자를 맨 뒤에 붙여서 새로운 3글자 세트를 만듭니다.
            context = torch.cat([context[:, 1:], ix], dim=1)

        results.append("".join(out))
    return results

sample(model, block_size, itos, device, num_samples=10)
# 이전 바이그램 모델보다 만들어진 이름들이 훨씬 더 그럴싸해진 것을 확인할 수 있습니다.
# (예: 'marva', 'kan', 'roy' 등 실제 이름처럼 보입니다.)

['chruet',
 'kyann',
 'jatie',
 'kalee',
 'rub',
 'madilis',
 'sil',
 'mil',
 'derimeth',
 'maruih']

## 6. 정리

- dataset은 `context -> next char` 구조를 유지합니다.
- 차이는 모델 내부입니다.
- embedding과 MLP를 사용하므로 bigram보다 더 풍부한 문맥을 처리할 수 있습니다.